# Gemini debug notebook

Этот блокнот нужен, чтобы посмотреть, что именно возвращает модель Gemini в вашем проекте, без Telegram-бота.

Шаги:
1. положите фото в корень проекта как `photo.jpg`;
2. запустите ячейки по порядку;
3. смотрите статус, объект и JSON-ответ модели.

In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'app.py').exists():
            return candidate
    return current


project_root = find_project_root()
print('Project root:', project_root)
print('Python executable:', sys.executable)
print('Python version:', sys.version.split()[0])

env_path = project_root / 'env'
if env_path.exists():
    load_dotenv(env_path)
else:
    load_dotenv(project_root / '.env')

venv_python = project_root / '.venv' / 'bin' / 'python'
print('Expected venv:', venv_python)

print('Environment loaded. GEMINI_API_KEY is set:', bool(os.getenv('GEMINI_API_KEY')))
#print('GEMINI_API_KEY value:', os.getenv('GEMINI_API_KEY'))

if venv_python.exists() and sys.executable != str(venv_python):
    print('\nWARNING: notebook kernel is not using the project .venv yet.')
    print('Select this kernel in VS Code and rerun the notebook:')
    print(venv_python)


Project root: /Users/qur1s/IT/Project/Fat secret sync
Python executable: /usr/local/bin/python3
Python version: 3.14.6
Expected venv: /Users/qur1s/IT/Project/Fat secret sync/.venv/bin/python
Environment loaded. GEMINI_API_KEY is set: True

Select this kernel in VS Code and rerun the notebook:
/Users/qur1s/IT/Project/Fat secret sync/.venv/bin/python


In [2]:
from pathlib import Path
import sys

# make the project root importable regardless of notebook location
project_root = Path.cwd().resolve()
for candidate in [project_root, *project_root.parents]:
    if (candidate / 'app.py').exists():
        project_root = candidate
        break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from clients.gemini_client import recognize_image
print('Imported recognize_image from project client')
print('Project root:', project_root)


Imported recognize_image from project client
Project root: /Users/qur1s/IT/Project/Fat secret sync


In [3]:
project_root = find_project_root()
image_path = project_root / 'photo.jpg'

if not image_path.exists():
    raise FileNotFoundError(
        f'Не найден файл {image_path.name}. Положите фото в корень проекта и назовите его {image_path.name}'
    )

image_bytes = image_path.read_bytes()
print(f'Loaded image: {image_path} ({len(image_bytes)} bytes)')

Loaded image: /Users/qur1s/IT/Project/Fat secret sync/photo.jpg (54682 bytes)


In [4]:
result = recognize_image(
    image_bytes=image_bytes,
    description="""\
    На фото может быть одно из трёх: готовое блюдо на тарелке или в миске;
    отдельный продукт (например, яблоко, банан, варёное яйцо, рожок мороженого,
    кусок пиццы, шоколадный батончик); или упаковка готовой еды.

    Нужно определить, что изображено, и его состав.

    - Блюдо: название блюда и его видимые компоненты, вес каждого компонента
    в граммах (разумная оценка по виду и размеру порции).
    - Отдельный продукт: сам продукт и его примерный вес или размер порции
    (одно среднее яблоко ≈ 180 г, один рожок мороженого ≈ 110 г, один банан
    без кожуры ≈ 120 г, один ломтик хлеба ≈ 30 г).
    - Упаковка: если на этикетке читаются название продукта, бренд,
    производитель, вес нетто или количество порций — использовать эти данные.
    Состав описывать по названию продукта, а не переписывать список
    ингредиентов с этикетки.

    Указывать только ингредиенты, физически видимые на фото. Не добавлять
    специи, приправы, масло, соусы и другие добавки, которых на фото не видно.
    Видимый соус или заправка (например, сливочный соус на пасте, глазурь на
    пончике) считается отдельным компонентом. Не включать тарелку, приборы,
    упаковку и любые несъедобные предметы.
    """,
    meal_type='lunch',
)

print(type(result))
print('status =', result.status)
print('meal_name =', result.meal_name)
print('brand =', result.brand)
print('items =', result.items)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


<class 'models.Meal.MealRecognition'>
status = MealStatus.ok
meal_name = Caesar salad
brand = None
items = [MealItem(name='Caesar salad', brand=None, amount_g=250)]


In [5]:
payload = result.model_dump(mode='json')
print(json.dumps(payload, ensure_ascii=False, indent=2))

{
  "status": "ok",
  "meal_kind": "single_item",
  "meal_name": "Caesar salad",
  "brand": null,
  "items": [
    {
      "name": "Caesar salad",
      "brand": null,
      "amount_g": 250
    }
  ]
}
